# Sentinel-2 and HLS seasonal mosaics of Guinea

Cloud-free mosaics of Guinea, **one per season** from two sources:

| | Sentinel-2 | HLS: Harmonized Landsat Sentinel-2 (NASA) |
|---|---|---|
| Resolution | 10 m (the 20 m bands are resampled) | 30 m |
| Sensors | Sentinel-2 | Landsat 8/9 **and** Sentinel-2, made interchangeable (same bands, BRDF- and bandpass-adjusted) |
| Available from | surface reflectance complete worldwide from December 2018 | 2013 (Landsat 8 only until Sentinel-2 starts, late 2015) |
| Cloud mask | [Cloud Score+](https://developers.google.com/earth-engine/datasets/catalog/GOOGLE_CLOUD_SCORE_PLUS_V1_S2_HARMONIZED) | HLS `Fmask` |
| Best for | detail: visual interpretation (reference labels of `lulc_benchmark.ipynb`), small clearings | more clear observations in the cloudy south; a consistent 30 m series since 2013, at the resolution of Hansen and the Landsat products, to build our own annual forest maps |

### Multiple seasons accounted

Guinea has one dry season (about November–April) and one rainy season (about May–October), and its vegetation looks
very different in each:

- **Dry season**: few clouds, so complete mosaics. Deciduous trees of the open forests and wooded savannas lose part
  of their leaves, grass is dry, and fires leave dark burn scars (mostly December–March) that can look like clearings.
  The harmattan brings dust haze in December–February.
- **Rainy season**: full leaf, green grass and growing crops, but many clouds, so fewer clear observations and
  possible gaps (section 5 measures them).

The **difference between the two seasons** separates land covers that look alike on one date: evergreen forest stays
green all year; deciduous woodland, savanna and cropland green up strongly in the rains. This matters for Guinea
because most of its forest under the national definition is open forest and wooded savanna (NERF 2026, stratum 2),
which a single dry-season image confuses with savanna and fallow. Section 6 shows this contrast.

## 1. Setup

In [1]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import ee
import geemap
import geopandas as gpd
import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd().parent                     # .../Guinea_Methodology

KEY_CANDIDATES = [
    PROJECT_ROOT / "gee-account.json",
    PROJECT_ROOT / "Land Use-Deforestation" / "landuse_tool" / "credentials" / "tribal-mapper-275801-7b57ce84f3bd.json",
]
KEY_FILE = next((path for path in KEY_CANDIDATES if path.exists()), None)
if KEY_FILE is None:
    raise FileNotFoundError(f"No Earth Engine key found, looked for: {[str(path) for path in KEY_CANDIDATES]}")
OUTPUT_DIR = Path.cwd() / "output_datasets"
ECOREGIONS_GPKG = OUTPUT_DIR / "ecoregions_guinea.gpkg"   # Guinea boundary, ecoregions and Sentinel-2 tiles
EXPORT_LOG_JSON = OUTPUT_DIR / "mosaic_exports.json"       # exports started, to follow them after a restart

c:\Users\ASUS\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.10.0). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.11, and then update google.api_core.
  warnings.warn(message, FutureWarning)


## 2. Parameters

If want to implement other mosaic series (e.g., 2014–2025 series of dry-season):

```python
SEASONS = dict(dry_season(year) for year in range(2014, 2026))
```

In [ ]:
def dry_season(year):
    """Dry season ending in `year`: 1 November of the year before → 30 April."""
    return f"dry_{year - 1}_{year}", (f"{year - 1}-11-01", f"{year}-05-01")

def rainy_season(year):
    """Rainy season of `year`: 1 May → 31 October."""
    return f"rainy_{year}", (f"{year}-05-01", f"{year}-11-01")

SEASONS = dict([dry_season(2025), rainy_season(2025)])
SEASONAL_CONTRAST = ("dry_2024_2025", "rainy_2025")  # (dry, rainy) seasons compared in section 6

MAX_SCENE_CLOUD_PERCENT = 70     # Sentinel-2 scenes cloudier than this are skipped before masking (saves compute)
S2_CLOUD_SCORE_BAND = "cs_cdf"   # Cloud Score+ band: "cs_cdf" (Earth Engine's example) or "cs" (investigate_tier.ipynb)
S2_CLEAR_THRESHOLD = 0.60        # keep pixels with a score ≥ this; higher = stricter
S2_BANDS = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]
S2_EXPORT_BANDS = ["B2", "B3", "B4", "B8", "B11", "B12"]  # blue, green, red, NIR, SWIR 1, SWIR 2: smaller exports
HLS_MASK_HIGH_AEROSOL = True     # also drop HLS pixels flagged as high aerosol (harmattan dust)

QUALITY_SCALE_M = 300            # resolution of the clear-observation statistics (section 5)

EXPORT_DESTINATION = "asset"     # "asset", "gcs" or "drive" (section 7)
ASSET_FOLDER = "guinea_mosaics"  # asset folder under projects/<GEE project>/assets/, or folder in the bucket / Drive
GCS_BUCKET = None                # bucket name, needed for "gcs"
SENSORS_TO_EXPORT = ["s2", "hls"]
SEASONS_TO_EXPORT = list(SEASONS)
PILOT_TILE = None                # None = the tile with the largest area inside Guinea
START_PILOT_EXPORT = False       # section 7.2
START_ALL_EXPORTS = False        # section 7.4: only after reading the pilot's cost in section 7.3

## 3. Connect to Earth Engine

In [3]:
with open(KEY_FILE) as f:
    key_info = json.load(f)

SERVICE_ACCOUNT = key_info["client_email"]
GEE_PROJECT = key_info["project_id"]

credentials = ee.ServiceAccountCredentials(SERVICE_ACCOUNT, key_file=str(KEY_FILE))
ee.Initialize(credentials=credentials, project=GEE_PROJECT)

guinea_boundary = gpd.read_file(ECOREGIONS_GPKG, layer="guinea_boundary")
guinea_ee = ee.Geometry(guinea_boundary.geometry.iloc[0].simplify(0.001).__geo_interface__)  # ~100 m simplification
guinea_bbox = ee.Geometry.Rectangle(list(guinea_boundary.total_bounds))

## 4. The mosaics

### 4.1 Sentinel-2

`COPERNICUS/S2_SR_HARMONIZED` (surface reflectance, with the 2022 processing offset already removed), masked with
Cloud Score+, which also catches cloud shadows and thin clouds better than the scene classification (SCL).

In [4]:
def sentinel2_collection(region, start, end):
    cloud_score = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")
    return (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(region)
        .filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", MAX_SCENE_CLOUD_PERCENT))
        .linkCollection(cloud_score, [S2_CLOUD_SCORE_BAND])
        .map(lambda image: image.updateMask(image.select(S2_CLOUD_SCORE_BAND).gte(S2_CLEAR_THRESHOLD)).select(S2_BANDS))
    )


def sentinel2_mosaic(region, start, end):
    """Median of the clear observations (uint16, reflectance × 10,000) and the number of clear observations."""
    collection = sentinel2_collection(region, start, end)
    clear_obs = collection.select("B4").count().unmask(0).rename("clear_obs")
    return collection.median().round().toUint16().addBands(clear_obs.toUint16())


for season, (start, end) in SEASONS.items():
    print(f"{season}: Sentinel-2 scenes over Guinea:", sentinel2_collection(guinea_bbox, start, end).size().getInfo())

dry_2024_2025: Sentinel-2 scenes over Guinea: 3880
rainy_2025: Sentinel-2 scenes over Guinea: 2532


### 4.2 HLS

The two HLS collections have different band numbers for the same wavelengths, so they are renamed to common names
before being merged. For Sentinel-2 the narrow NIR band (`B8A`) is used, as HLS does, because it matches Landsat's NIR.

`Fmask` bits used: 1 cloud, 2 next to cloud or shadow, 3 cloud shadow; bits 6–7 = 3 is high aerosol.

The scale of the HLS values on Earth Engine (stored as reflectance × 10,000, or as reflectance) is checked on one
image rather than assumed. On Earth Engine they are reflectance (scale 1, checked on 19 October 2026).

In [5]:
HLS_BANDS = ["blue", "green", "red", "nir", "swir1", "swir2"]
HLS_SOURCES = {
    "NASA/HLS/HLSL30/v002": ["B2", "B3", "B4", "B5", "B6", "B7"],     # Landsat 8/9
    "NASA/HLS/HLSS30/v002": ["B2", "B3", "B4", "B8A", "B11", "B12"],  # Sentinel-2 (narrow NIR, like Landsat)
}

probe_region = ee.Geometry.Point([-13.6, 9.6]).buffer(5000)  # near Conakry
probe = ee.ImageCollection("NASA/HLS/HLSL30/v002").filterBounds(probe_region).filterDate("2020-01-01", "2026-01-01").first()
probe_red = probe.select("B4").reduceRegion(ee.Reducer.median(), probe_region, 30).get("B4").getInfo()
HLS_SCALE = 0.0001 if probe_red > 2 else 1  # values > 2 cannot be reflectance: they are stored × 10,000
print(f"HLS red near Conakry: {probe_red:.4f} → HLS_SCALE = {HLS_SCALE}")


def hls_masker(source_bands):
    """Function that masks one HLS image with its Fmask and renames its bands to HLS_BANDS (reflectance × 10,000)."""
    def mask(image):
        fmask = image.select("Fmask")
        bad = fmask.bitwiseAnd(0b1110).neq(0)  # bit 1 cloud, bit 2 next to cloud/shadow, bit 3 cloud shadow
        if HLS_MASK_HIGH_AEROSOL:
            bad = bad.Or(fmask.rightShift(6).bitwiseAnd(0b11).eq(3))
        return image.select(source_bands, HLS_BANDS).multiply(HLS_SCALE * 10_000).updateMask(bad.Not())
    return mask


def hls_collection(region, start, end):
    landsat, sentinel = [
        ee.ImageCollection(asset).filterBounds(region).filterDate(start, end).map(hls_masker(bands))
        for asset, bands in HLS_SOURCES.items()
    ]
    return landsat.merge(sentinel)


def hls_mosaic(region, start, end):
    """Median of the clear observations (int16, reflectance × 10,000) and the number of clear observations."""
    collection = hls_collection(region, start, end)
    clear_obs = collection.select("red").count().unmask(0).rename("clear_obs")
    return collection.median().round().toInt16().addBands(clear_obs.toInt16())


for season, (start, end) in SEASONS.items():
    counts = {asset.split("/")[2]: ee.ImageCollection(asset).filterBounds(guinea_bbox).filterDate(start, end).size().getInfo()
              for asset in HLS_SOURCES}
    print(f"{season}: HLS scenes over Guinea:", counts)

HLS red near Conakry: 0.0908 → HLS_SCALE = 1
dry_2024_2025: HLS scenes over Guinea: {'HLSL30': 2639, 'HLSS30': 4498}
rainy_2025: HLS scenes over Guinea: {'HLSL30': 1928, 'HLSS30': 4567}


## 5. Quality: how many clear observations?

For each ecoregion and season: the mean number of clear observations, and the share of the area with a gap or fewer than 3.

The rainy season is where gaps are expected. A season with many gaps (or many pixels under 3 observations) cannot be used alone.

In [6]:
def get_info_with_eecu(request):
    """Compute an Earth Engine object and return (result, seconds, eecu_seconds).

    eecu_seconds is None if the profile could not be fetched (it is sometimes not ready on the server yet).
    """
    profile_ids = []
    start = time.perf_counter()
    with ee.data.profiling(profile_ids.append):
        result = request.getInfo()
    seconds = time.perf_counter() - start

    get_profiles = ee.ApiFunction.lookup("Profile.getProfiles").call
    for attempt in range(5):
        try:
            profile_text = get_profiles(ids=profile_ids).getInfo()
        except ee.EEException:
            time.sleep(2 * (attempt + 1))
            continue
        # The profile is a table; its first column is EECU-seconds per operation ("-" when negligible).
        eecu_seconds = sum(
            float(line.split()[0]) for line in profile_text.splitlines()[1:] if line.split() and line.split()[0] != "-"
        )
        return result, seconds, round(eecu_seconds, 1)
    return result, seconds, None


MOSAICS = {"s2": sentinel2_mosaic, "hls": hls_mosaic}
ecoregions = gpd.read_file(ECOREGIONS_GPKG, layer="ecoregions_guinea")
zones = ee.FeatureCollection(
    [ee.Feature(ee.Geometry(geometry.simplify(0.005).__geo_interface__), {"zone": name})
     for name, geometry in zip(ecoregions["ECO_NAME"], ecoregions.geometry)]
    + [ee.Feature(guinea_ee, {"zone": "Guinea"})]
)


def season_quality(start, end):
    rows = []
    for sensor, mosaic in MOSAICS.items():
        clear = mosaic(guinea_bbox, start, end).select("clear_obs")
        stats_image = ee.Image.cat([clear.rename("mean_clear_obs"), clear.eq(0).rename("no_clear_obs"),
                                    clear.lt(3).rename("under_3_clear_obs")])
        request = stats_image.reduceRegions(collection=zones, reducer=ee.Reducer.mean(), scale=QUALITY_SCALE_M, tileScale=4)
        result, seconds, eecu_seconds = get_info_with_eecu(request)
        print(f"  {sensor}: {seconds:.0f} s, {eecu_seconds} EECU-seconds")
        rows += [{"sensor": sensor, **feature["properties"], "eecu_seconds": eecu_seconds} for feature in result["features"]]
    table = pd.DataFrame(rows)
    table["no_clear_obs_%"] = 100 * table.pop("no_clear_obs")
    table["under_3_clear_obs_%"] = 100 * table.pop("under_3_clear_obs")
    return table


tables = []
for season, (start, end) in SEASONS.items():
    path = OUTPUT_DIR / f"mosaic_quality_{season}.parquet"
    if path.exists():
        table = pd.read_parquet(path)
        print(f"{season}: read {path.name} (no Earth Engine compute used)")
    else:
        print(f"{season}: computing")
        table = season_quality(start, end)
        table.to_parquet(path, index=False)
    tables.append(table.assign(season=season))
quality = pd.concat(tables, ignore_index=True)

for column, title in [("mean_clear_obs", "Mean number of clear observations per pixel"),
                      ("no_clear_obs_%", "Area without any clear observation (%)")]:
    fig = px.bar(quality, x="zone", y=column, color="sensor", barmode="group", facet_row="season",
                 labels={column: "", "zone": ""}, title=title, template="plotly_white", height=300 * len(SEASONS))
    fig.show()
quality.pivot_table(index="zone", columns=["season", "sensor"], values=["mean_clear_obs", "no_clear_obs_%"]).round(1)

dry_2024_2025: read mosaic_quality_dry_2024_2025.parquet (no Earth Engine compute used)
rainy_2025: read mosaic_quality_rainy_2025.parquet (no Earth Engine compute used)


mean_clear_obs                         \
season                           dry_2024_2025       rainy_2025         
sensor                                     hls    s2        hls    s2   
zone                                                                    
Guinea                                    35.8  43.0       12.2  19.9   
Guinean forest-savanna                    36.6  44.9       10.7  19.3   
Guinean mangroves                         27.4  42.0        6.7  15.7   
Guinean montane forests                   46.2  43.7       12.3  19.0   
West Sudanian savanna                     34.1  42.0       19.0  26.9   
Western Guinean lowland forests           23.8  32.8        3.6   6.9   

                                no_clear_obs_%                       
season                           dry_2024_2025      rainy_2025       
sensor                                     hls   s2        hls   s2  
zone                                                                 
Guinea                                     0.0  0.0        1.6  0.1  
Guinean forest-savanna                     0.0  0.0        0.4  0.0  
Guinean mangroves                          0.0  0.0        3.4  0.0  
Guinean montane forests                    0.0  0.0        0.3  0.0  
West Sudanian savanna                      0.0  0.0        0.0  0.0  
Western Guinean lowland forests            0.3  0.0       15.5  1.1

## 6. Look at the mosaics

False colour (SWIR 1 / NIR / red): vegetation is green, bare soil and built-up areas pink-purple, water dark blue, burn scars dark brown.

**Seasonal greening** (HLS NDVI of the rainy season minus the dry season, from `SEASONAL_CONTRAST`): white = about
the same in both seasons (evergreen forest, or permanently bare / built / water), red = much greener in the rains
(deciduous woodland, savanna, crops). This is the information a single season does not have.

In [7]:
TRUE_COLOUR = {"s2": ["B4", "B3", "B2"], "hls": ["red", "green", "blue"]}
FALSE_COLOUR = {"s2": ["B11", "B8", "B4"], "hls": ["swir1", "nir", "red"]}
SENSOR_NAMES = {"s2": "Sentinel-2", "hls": "HLS"}
clear_obs_vis = {"bands": ["clear_obs"], "min": 0, "max": 30, "palette": ["#d7191c", "#fdae61", "#ffffbf", "#1a9641"]}

m = geemap.Map(center=[10.4, -11], zoom=7)
first_season = next(iter(SEASONS))
for season, (start, end) in SEASONS.items():
    for sensor, mosaic in MOSAICS.items():
        image = mosaic(guinea_bbox, start, end).clip(guinea_ee)
        name = f"{SENSOR_NAMES[sensor]} {season}"
        m.add_layer(image, {"bands": FALSE_COLOUR[sensor], "min": 0, "max": 4500}, f"{name} false colour", False)
        m.add_layer(image, {"bands": TRUE_COLOUR[sensor], "min": 0, "max": 2500}, f"{name} true colour",
                    season == first_season and sensor == "s2")
        m.add_layer(image, clear_obs_vis, f"{name} clear observations", False)

dry, rainy = SEASONAL_CONTRAST
if dry in SEASONS and rainy in SEASONS:
    def hls_ndvi(season):
        return hls_mosaic(guinea_bbox, *SEASONS[season]).normalizedDifference(["nir", "red"])
    greening = hls_ndvi(rainy).subtract(hls_ndvi(dry)).clip(guinea_ee)
    m.add_layer(greening, {"min": 0, "max": 0.4, "palette": ["#f7f7f7", "#fdae61", "#d7191c"]},
                f"Seasonal greening: NDVI {rainy} − {dry} (HLS)", False)
m.add_layer(ee.FeatureCollection([ee.Feature(guinea_ee)]).style(color="white", fillColor="00000000", width=2), {}, "Guinea")
m

Map(center=[10.4, -11], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', t…

## 8. Notes for the deforestation methodology

**What the runs showed** (dry season 2024–2025 and rainy season 2025, October 2026), section 5:

| Mean clear observations per pixel | Dry, Sentinel-2 | Dry, HLS | Rainy, Sentinel-2 | Rainy, HLS |
|---|---|---|---|---|
| Guinea | 43 | 36 | 20 | 12 |
| Western Guinean lowland forests | 33 | 24 | 7 | 4 |
| Area with no clear observation in the lowland forests | 0 % | 0.3 % | 1.1 % | 15.5 % |

- **The dry season gives complete mosaics everywhere**: it is the base for annual forest maps.
- **The rainy season is usable in the north and centre, but not alone in the western lowland forests**, the main
  deforestation hotspot of Hansen, where HLS leaves 15 % gaps and Sentinel-2 averages 7 clear observations. Use rainy
  inputs there only as extra features where they exist, or widen the season.
- **HLS has fewer clear observations than Sentinel-2**, although it adds Landsat: its Fmask and aerosol masks are
  stricter than Cloud Score+. Before Sentinel-2 joins HLS (late 2015) there is only Landsat 8: check section 5 for the
  earliest years before relying on them.

**How these mosaics can improve the deforestation method**

1. **Our own consistent forest-change series.** Hansen's series breaks in 2013 (`hansen_annual_maps.ipynb`). A
   series of dry-season HLS mosaics from 2014 to 2025 (`SEASONS = dict(dry_season(y) for y in range(2014, 2026))`)
   is made the same way every year. Classifying forest / non-forest on each year with the same model, trained on
   reference points under the national definition, gives an annual series without a method break. Using all
   observations instead of seasonal medians, continuous change detection (CCDC, Zhu & Woodcock 2014) is the
   established method for this, and it is available on Earth Engine.
2. **Seasonal contrast for Guinea's open forests.** Most forest under the national definition (> 10 % canopy) is open
   forest and wooded savanna, which one dry-season image confuses with savanna and fallow. Using a dry **and** a
   rainy mosaic (section 6, seasonal greening) as inputs separates evergreen forest, deciduous woodland, savanna and
   crops better; time series of images are known to improve land-cover classification over single dates (Gómez et al.
   2016).
3. **Fire is not deforestation.** Dry-season burn scars (mostly December–March) look like clearings in a late
   dry-season mosaic. Prefer early dry season dates (November–January) for change detection, or mask burned areas
   (MODIS burned area, already listed in `deforestation_tool/config.py`).
4. **Reference labelling.** The 10 m Sentinel-2 mosaic of the comparison year is one of the images interpreters
   should look at for the reference points of `lulc_benchmark.ipynb` (section 10), next to very high resolution
   imagery (Google Earth Pro, Esri Wayback).
5. **Export only what the method needs.** One season of 10 m Sentinel-2 for all of Guinea is ~44 GB uncompressed;
   HLS is ~5 GB. For classification on Earth Engine nothing needs exporting; exports are for working outside it.

**References**

- Gómez, C., White, J. C., & Wulder, M. A. (2016). Optical remotely sensed time series data for land cover
  classification: A review. *ISPRS Journal of Photogrammetry and Remote Sensing*, 116, 55–72.
- Zhu, Z., & Woodcock, C. E. (2014). Continuous change detection and classification of land cover using all
  available Landsat data. *Remote Sensing of Environment*, 144, 152–171.
- République de Guinée (2026). *NRF / NERF de la Guinée, période 2016–2020*, submitted to the UNFCCC.